# Skin Cancer Detection — ISIC 2024
## Notebook 4: Out-of-Fold Target Encoding

**Goal:** Encode categorical columns without data leakage.

**Why OOF:** Global encoding leaks the answer into training data.
OOF encoding calculates each fold's encoding using OTHER folds only — never itself.

---
### Notebook Structure:
- **Week 3** — Understanding Data Leakage (why global encoding fails)
- **Week 4** — Building a reusable OOF encoder function

##  Step 1 — Imports and Load Data

We load the cleaned dataset produced in Notebook 2.

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

data = pd.read_csv('/content/cleaned_train_metadata (1).csv')
print("Shape:", data.shape)
print("Cancer cases:", data['target'].sum())
print("Missing values:", data.isnull().sum().sum())

Shape: (401059, 39)
Cancer cases: 393
Missing values: 0


## Step 2 — Split X and y

We separate features (X) from target (y) before building folds.

In [11]:
y = data['target']
X = data.drop(columns=['target'])

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Cancer cases:", y.sum())
print("Cancer rate:", round(y.mean() * 100, 4), "%")

X shape: (401059, 38)
y shape: (401059,)
Cancer cases: 393
Cancer rate: 0.098 %


---
##  WEEK 3 — Understanding Data Leakage

### The Problem With Global Target Encoding

When we encode a categorical column like `anatom_site_general`,
the naive approach is to calculate the cancer rate for each site
using ALL patients — then apply it to those SAME patients.

This is called **Data Leakage** — the model sees the answer
before it makes a prediction.

### First — Let's Build The OOF Loop (Correct Way)

Before showing WHY global encoding fails, we first build
the correct OOF encoding so we can compare both side by side.

In [12]:
# OOF Target Encoding Loop — First Version
# Encoding 'anatom_site_general' column

data['anatom_site_encoded'] = np.nan

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold = 1

for train_idx, validation_idx in skf.split(X, y):
    train_data = data.iloc[train_idx]
    val_data = data.iloc[validation_idx]

    # Calculate cancer rate per site using ONLY training rows
    encoding_map = train_data.groupby(
        'anatom_site_general')['target'].mean()

    # Apply to validation rows only
    data.loc[data.index[validation_idx], 'anatom_site_encoded'] =         val_data['anatom_site_general'].map(encoding_map)

    print(f"Fold {fold} encoded!")
    fold += 1

print("\nMissing values:", data['anatom_site_encoded'].isnull().sum())

Fold 1 encoded!
Fold 2 encoded!
Fold 3 encoded!
Fold 4 encoded!
Fold 5 encoded!

Missing values: 0


###  Simulation — Global Encoding vs OOF Encoding

Now we compare global encoding (wrong) vs OOF encoding (correct)
side by side to prove that global encoding leaks information.

In [13]:
# GLOBAL ENCODING — WRONG WAY ❌
global_encoding = data.groupby(
    'anatom_site_general')['target'].mean()

data['global_encoded'] = data['anatom_site_general'].map(
    global_encoding)

print("=== GLOBAL ENCODING (LEAKY) ===")
print(data[['anatom_site_general',
            'global_encoded']].drop_duplicates())

print("\n=== OOF ENCODING (CORRECT) ===")
print(data[['anatom_site_general',
            'anatom_site_encoded']].drop_duplicates())

print("\n⚠️ Global encoding uses ALL rows to calculate")
print("rates — including the rows being predicted.")
print("This leaks the answer into training! ❌")
print("\n✅ OOF encoding calculates rates using OTHER")
print("folds only — never the row being predicted.")

=== GLOBAL ENCODING (LEAKY) ===
  anatom_site_general  global_encoded
0     lower extremity        0.000709
1           head/neck        0.006475
2     posterior torso        0.000807
3      anterior torso        0.000934
6     upper extremity        0.000808

=== OOF ENCODING (CORRECT) ===
    anatom_site_general  anatom_site_encoded
0       lower extremity             0.000691
1             head/neck             0.006193
2       posterior torso             0.000802
3        anterior torso             0.000909
4        anterior torso             0.000941
6       upper extremity             0.000780
7       posterior torso             0.000756
8        anterior torso             0.000939
9       upper extremity             0.000743
10      lower extremity             0.000692
11      posterior torso             0.000832
13      upper extremity             0.000761
14      posterior torso             0.000823
15       anterior torso             0.000981
16      upper extremity          

## Conceptual Map — Why OOF Encoding Prevents Leakage

### Global Encoding (WRONG) ❌:
```
All 401,059 patients → calculate cancer rate
→ apply to SAME 401,059 patients
→ model sees its own answer during training ❌
```

### Out-of-Fold Encoding (CORRECT) ✅:
```
Fold 1 patients → encoded using Folds 2,3,4,5 ✅
Fold 2 patients → encoded using Folds 1,3,4,5 ✅
Fold 3 patients → encoded using Folds 1,2,4,5 ✅
Fold 4 patients → encoded using Folds 1,2,3,5 ✅
Fold 5 patients → encoded using Folds 1,2,3,4 ✅
No patient ever sees its own answer! ✅
```

### Why Different Values Per Fold?
Global encoding gives ONE fixed value per site.
OOF encoding gives SLIGHTLY different values per fold
because each fold uses different training data.
This variation PROVES there is no leakage. ✅

### Key Insight:
```
Global encoding → inflated scores → fails on real patients ❌
OOF encoding   → honest scores  → works on real patients  ✅
```

---
## WEEK 4 — Reusable OOF Encoder Function

### The Problem With Our Current Loop

Our Week 3 loop only encodes `anatom_site_general`.
If we want to encode another column like `tbp_lv_location`,
we would have to copy-paste the entire loop.

### The Solution — A Reusable Function

We wrap the encoding logic into a clean function that:
- Works for **ANY** categorical column ✅
- Takes the column name as a parameter ✅
- Returns the encoded dataframe ✅

In [14]:
def oof_target_encode(data, column, target, n_splits=5):
    """
    Encodes a categorical column using Out-of-Fold target encoding.

    Parameters:
        data     : Full dataframe
        column   : Categorical column to encode
        target   : Target column name (e.g. 'target')
        n_splits : Number of folds (default 5)

    Returns:
        data with new encoded column added
    """
    y = data[target]
    X = data.drop(columns=target)

    encoded_col_name = column + '_encoded'
    data[encoded_col_name] = np.nan

    skf = StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=42)

    fold = 1
    for train_idx, validation_idx in skf.split(X, y):
        train_data = data.iloc[train_idx]
        val_data = data.iloc[validation_idx]

        encoding_map = train_data.groupby(column)[target].mean()

        data.loc[data.index[validation_idx], encoded_col_name] = val_data[column].map(encoding_map)

        print(f"Fold {fold} encoded!")
        fold += 1

    return data

print("✅ Function defined successfully!")

✅ Function defined successfully!


## Step 3 — Apply OOF Encoding To All Categorical Columns

We apply our reusable function to all high-cardinality categorical columns.

In [15]:
# Apply OOF encoding to anatom_site_general
print("Encoding anatom_site_general...")
data = oof_target_encode(data, 'anatom_site_general', 'target')
print("Missing values:", data['anatom_site_general_encoded'].isnull().sum())
print()

# Apply OOF encoding to tbp_lv_location
print("Encoding tbp_lv_location...")
data = oof_target_encode(data, 'tbp_lv_location', 'target')
print("Missing values:", data['tbp_lv_location_encoded'].isnull().sum())

Encoding anatom_site_general...
Fold 1 encoded!
Fold 2 encoded!
Fold 3 encoded!
Fold 4 encoded!
Fold 5 encoded!
Missing values: 0

Encoding tbp_lv_location...
Fold 1 encoded!
Fold 2 encoded!
Fold 3 encoded!
Fold 4 encoded!
Fold 5 encoded!
Missing values: 0


##  Step 4 — Label Encode Remaining Categorical Columns

For low-cardinality columns like `sex` (only 2 values),
we use simple label encoding instead of OOF encoding.

In [16]:
# Label encode sex column (male/female → 0/1)
data['sex'] = data['sex'].astype('category').cat.codes
print("Sex encoded:", data['sex'].value_counts().to_dict())

# Check remaining text columns
remaining_text = data.select_dtypes(include=['object']).columns.tolist()
print("\nRemaining text columns:", remaining_text)

Sex encoded: {1: 277063, 0: 123996}

Remaining text columns: ['anatom_site_general', 'tbp_lv_location']


##  Step 5 — Drop Original Text Columns

Now that we have encoded versions, we drop the original
text columns — models can only work with numbers.

In [17]:
# Drop original text columns we already encoded
data = data.drop(columns=['anatom_site_general', 'tbp_lv_location'])

# Drop global_encoded column (it was just for simulation — leaky!)
if 'global_encoded' in data.columns:
    data = data.drop(columns=['global_encoded'])

print("Shape after dropping text columns:", data.shape)
print("Remaining text columns:",
      data.select_dtypes(include=['object']).columns.tolist())

Shape after dropping text columns: (401059, 40)
Remaining text columns: []


##  Step 6 — Save Encoded Dataset For Week 5

We save the fully encoded dataset so Notebook 5
(LightGBM Training) can load it directly.

In [18]:
# Final verification
print("=== FINAL ENCODED DATASET SUMMARY ===")
print(f"Shape: {data.shape}")
print(f"Cancer cases: {data['target'].sum()}")
print(f"Missing values: {data.isnull().sum().sum()}")
print(f"All numerical: {data.select_dtypes(include=['object']).shape[1] == 0}")

# Save to CSV
data.to_csv('/content/encoded_train_metadata.csv', index=False)
print("\n✅ Saved as encoded_train_metadata.csv")
print("Ready for Week 5 — LightGBM Training!")

=== FINAL ENCODED DATASET SUMMARY ===
Shape: (401059, 40)
Cancer cases: 393
Missing values: 0
All numerical: True

✅ Saved as encoded_train_metadata.csv
Ready for Week 5 — LightGBM Training!
